In [1]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [2]:
ds = pd.read_csv("smartphone-usage-and-addiction-prediction/versions/1/Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv")
ds = ds.drop(columns=["transaction_id","user_id"])
ds = pd.DataFrame(ds)
ds['addiction_level'] = ds['addiction_level'].fillna('none')
ds.head()
dd = ds.drop(columns=["daily_screen_time_hours"])

In [3]:
#seprating the input and the output in the data
X = ds.drop(columns=["addicted_label"])
y = ds["addicted_label"]

In [4]:
#spliting the data in 80% adnd 20%
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

In [5]:
X_train.shape,X_test.shape

((6000, 13), (1500, 13))

In [6]:
#searching for the best parameters 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}
model1 = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),              # plain RandomForestClassifier here
    param_distributions=param_dist,
    n_iter=30,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

In [7]:
#training the model with the data
model1.fit(X_train, y_train)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


ValueError: 
All the 90 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
30 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\MADHUSUDAN\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MADHUSUDAN\anaconda3\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\MADHUSUDAN\anaconda3\Lib\site-packages\sklearn\ensemble\_forest.py", line 334, in fit
    X, y = validate_data(
           ~~~~~~~~~~~~~^
        self,
        ^^^^^
    ...<5 lines>...
        ensure_all_finite=False,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
 